# TCC — Object Discovery Lab no Google Colab

Este notebook prepara o projeto para Colab com GPU, mantém `data/`, `workspace/`, `exports/` e `weights/` no Google Drive e permite:
- clustering das bounding boxes já anotadas;
- baseline SAM;
- abrir a interface Streamlit por um túnel ngrok opcional.

Antes de executar, selecione **Runtime > Change runtime type > GPU**.

Em **Colab > Secrets**, crie:
- `GITHUB_TOKEN` — token com leitura do repositório privado;
- `ROBOFLOW_API_KEY`;
- `NGROK_AUTHTOKEN` — opcional, apenas para abrir o Streamlit.


In [ ]:
import os, subprocess, sys, torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("VRAM:", round(props.total_memory / 1024**3, 2), "GB")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUNTIME_ROOT = "/content/drive/MyDrive/TCC_Code_Test_runtime"
os.makedirs(RUNTIME_ROOT, exist_ok=True)
for name in ["data", "workspace", "exports", "weights"]:
    os.makedirs(os.path.join(RUNTIME_ROOT, name), exist_ok=True)
print("Runtime persistente:", RUNTIME_ROOT)


In [ ]:
# Clona/atualiza o repositório privado sem imprimir o token.
from google.colab import userdata
from pathlib import Path
import os, subprocess

REPO_DIR = Path("/content/TCC_Code_Test")
token = userdata.get("GITHUB_TOKEN")

netrc = Path.home() / ".netrc"
netrc.write_text(
    f"machine github.com\nlogin x-access-token\npassword {token}\n",
    encoding="utf-8",
)
netrc.chmod(0o600)

try:
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(
            ["git", "clone", "https://github.com/ferreis/TCC_Code_Test.git", str(REPO_DIR)],
            check=True,
        )
finally:
    netrc.unlink(missing_ok=True)

os.chdir(REPO_DIR)
print("Repositório:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
%%capture
!pip install -r requirements-colab.txt


In [ ]:
# Liga as pastas pesadas ao Drive para sobreviver a reinícios do Colab.
from pathlib import Path
import shutil, os

repo = Path("/content/TCC_Code_Test")
runtime = Path("/content/drive/MyDrive/TCC_Code_Test_runtime")

for name in ["data", "workspace", "exports", "weights"]:
    target = runtime / name
    link = repo / name
    if link.is_symlink() or link.exists():
        if link.is_symlink():
            link.unlink()
        elif link.is_dir():
            # preserva conteúdo local copiando para o Drive, se houver
            for p in link.iterdir():
                dst = target / p.name
                if not dst.exists():
                    if p.is_dir():
                        shutil.copytree(p, dst)
                    else:
                        shutil.copy2(p, dst)
            shutil.rmtree(link)
        else:
            link.unlink()
    link.symlink_to(target, target_is_directory=True)

print("Pastas persistentes conectadas ao Drive.")


In [ ]:
# Configura o Roboflow via Secrets, sem gravar a chave no notebook.
from google.colab import userdata
import os

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
os.environ["ROBOFLOW_WORKSPACE"] = "rafael-fernando-dos-reis-mecabo"
os.environ["ROBOFLOW_PROJECT"] = "detecao-de-cetaceos-afrzs-iqey9"
os.environ["ROBOFLOW_VERSION"] = "1"

print("Roboflow configurado.")


In [ ]:
# Baixa o dataset apenas se ele ainda não existir no Drive.
from pathlib import Path
import subprocess, sys

images_dir = Path("data/roboflow/train/images")
has_images = images_dir.exists() and any(images_dir.iterdir())

if has_images:
    print("Dataset já existe:", images_dir)
else:
    subprocess.run(
        [
            sys.executable, "scripts/download_roboflow.py",
            "--format", "yolo26",
            "--output", "data/roboflow",
            "--overwrite",
        ],
        check=True,
    )


In [ ]:
# Baixa o SAM ViT-B apenas se o checkpoint ainda não estiver no Drive.
from pathlib import Path
import subprocess, sys

sam = Path("weights/sam_vit_b_01ec64.pth")
if sam.exists():
    print("SAM já existe:", sam)
else:
    subprocess.run([sys.executable, "scripts/download_sam.py"], check=True)


## Fluxo recomendado: clusterizar as bounding boxes já revisadas

Isto **não usa tiles para formar o objeto**. Cada box existente vira um crop completo com contexto, depois DINOv2 + HDBSCAN agrupam os cetáceos.


In [ ]:
from pathlib import Path
from tcc_discovery.box_clustering import BoxClusteringConfig, run_box_clustering

project_dir = Path("workspace/bbox_clusters_colab")
catalog_file = project_dir / "catalog.json"

if catalog_file.exists():
    print("Projeto já existe:", catalog_file)
    print("Abra o Streamlit para revisar ou altere project_dir para criar outro experimento.")
else:
    cfg = BoxClusteringConfig(
        context_scale=1.8,
        min_crop_size_px=64,
        min_cluster_size=6,
        embedding_batch_size=8 if torch.cuda.is_available() else 2,
        dino_cpu_fallback_on_oom=True,
    )
    result = run_box_clustering(
        "data/roboflow/train",
        project_dir,
        cfg,
        overwrite=False,
        progress_cb=lambda done, total, msg: print(f"[{done}/{total}] {msg}") if done % 100 == 0 or done == total else None,
    )
    print("Imagens:", len(result["images"]))
    print("Objetos:", len(result["objects"]))


## Interface Streamlit no Colab — opcional

Para usar a mesma interface gráfica, adicione `NGROK_AUTHTOKEN` em **Colab Secrets** e execute a célula abaixo.  
Os projetos/datasets continuam no Drive mesmo se a sessão Colab for encerrada.


In [ ]:
from google.colab import userdata
from pyngrok import ngrok
import subprocess, os, time

ngrok.kill()
ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

time.sleep(3)
public_url = ngrok.connect(8501, bind_tls=True).public_url
print("Abra a interface:", public_url)
print("Processo Streamlit PID:", streamlit_process.pid)


### Observações

- O **Colab pode desconectar** sessões longas. Arquivos em `/content` somem, por isso este notebook usa o Google Drive para `data`, `workspace`, `exports` e `weights`.
- Para o seu fluxo principal, use `workspace/bbox_clusters_colab`.
- O baseline SAM pode ser executado pela interface, mas para imagens de drone muito grandes mantenha auto-tiling e fallback para CPU ativos.
- Não coloque tokens diretamente nas células. Use **Colab Secrets**.
